# Lesson 4: State Reducers — Accumulating Results

## Objective
Use **reducers** (via `Annotated`) to safely accumulate multiple computation results into a history list, even when different invocations write to the same state field.

## Limitation of Lesson 3
- ❌ `result` was overwritten every call — previous computations lost
- ❌ No way to collect a history of calculations
- ❌ If two parallel nodes write to the same field, the last one wins (merge conflict)

## What is NEW in Lesson 4?
- ✅ `Annotated[T, reducer_fn]` — attach a reducer function to a state field
- ✅ `operator.add` as a list reducer (list + list = concatenation)
- ✅ Custom reducer functions for fine-grained merge control
- ✅ Understanding LangGraph's default merge strategy vs reducer-controlled merge

## Theory: Default Merge vs Reducer

### Default Merge (no reducer)
```python
current = {"history": ["5+3=8"]}
node returns {"history": ["4×2=8"]}
merged = {"history": ["4×2=8"]}   # ← OVERWRITTEN! "5+3=8" is LOST
```

### With Reducer: `Annotated[list, operator.add]`
```python
current = {"history": ["5+3=8"]}
node returns {"history": ["4×2=8"]}
LangGraph calls: operator.add(["5+3=8"], ["4×2=8"])
merged = {"history": ["5+3=8", "4×2=8"]}   # ← APPENDED!
```

`operator.add` on lists = **concatenation**.  
This is exactly how `MessagesState` works in LangGraph's built-in message accumulation (Lesson 6+).

## Architectural Importance
Reducers are the foundation of:
- **Message history** in LLM conversations (Lesson 6+)
- **Parallel result aggregation** (Lesson 9)
- **Map-reduce** collection of results (Lesson 10)


In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
import operator
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, Annotated, Literal
from IPython.display import Image, display

# operator.add on lists = list concatenation
# Example: operator.add(["a"], ["b"]) → ["a", "b"]
print("✓ Imports successful")
print(f"  operator.add([1,2], [3]) = {operator.add([1,2], [3])}")


In [ ]:
# ── Cell 2: State with Reducer ──────────────────────────────────────────────
# Annotated[list, operator.add] means:
#   - The field holds a list
#   - When a node updates this field, LangGraph calls:
#       operator.add(existing_value, returned_value)
#   - So: ["old"] + ["new"] = ["old", "new"]  ← APPEND, never overwrite

class State(TypedDict):
    a: int
    b: int
    operation: str
    result: Optional[float]                    # Plain field: overwritten each time
    history: Annotated[list, operator.add]     # REDUCER field: APPENDED each time

print("✓ State with reducer defined")
print()
print("  'result'  → plain field (last write wins)")
print("  'history' → reducer field (operator.add appends, never overwrites)")


In [ ]:
# ── Cell 3: Nodes — Each Appends to History ─────────────────────────────────
# Each node returns {"history": ["single entry as a list"]}
# LangGraph calls operator.add(existing_history, ["single entry"])
# Result: the entry is appended to the growing history list

def add_node(state: State) -> dict:
    res = state["a"] + state["b"]
    entry = f"{state['a']} + {state['b']} = {res}"
    print(f"  [add]      {entry}")
    return {
        "result": float(res),
        "history": [entry]     # ← Must be a LIST; reducer does list + list
    }

def multiply_node(state: State) -> dict:
    res = state["a"] * state["b"]
    entry = f"{state['a']} × {state['b']} = {res}"
    print(f"  [multiply] {entry}")
    return {
        "result": float(res),
        "history": [entry]
    }

def divide_node(state: State) -> dict:
    if state["b"] == 0:
        entry = f"{state['a']} ÷ {state['b']} = ERROR (div by zero)"
        return {"result": 0.0, "history": [entry]}
    res = state["a"] / state["b"]
    entry = f"{state['a']} ÷ {state['b']} = {res:.4f}"
    print(f"  [divide]   {entry}")
    return {
        "result": res,
        "history": [entry]
    }

print("✓ Three nodes defined with history accumulation")


In [ ]:
# ── Cell 4: Router ──────────────────────────────────────────────────────────
def router(state: State) -> Literal["add", "multiply", "divide"]:
    return state["operation"]

print("✓ Router defined")


In [ ]:
# ── Cell 5: Build Graph ─────────────────────────────────────────────────────
builder = StateGraph(State)

builder.add_node("add",      add_node)
builder.add_node("multiply", multiply_node)
builder.add_node("divide",   divide_node)

# Direct conditional routing from START (no pass-through node needed)
builder.add_conditional_edges(START, router, {
    "add":      "add",
    "multiply": "multiply",
    "divide":   "divide",
})

builder.add_edge("add",      END)
builder.add_edge("multiply", END)
builder.add_edge("divide",   END)

graph = builder.compile()
print("✓ Graph with reducer compiled")


In [ ]:
# ── Cell 6: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 7: Demonstrate Reducer Accumulation ─────────────────────────────────
# Each invoke() starts fresh — but we manually chain history to show accumulation
# In Lesson 17 (checkpointing), this happens automatically across invocations

print("Simulating session with accumulating history:")
print()
history_so_far = []  # We manually pass history between calls

operations = [
    {"a": 10, "b": 3, "op": "add"},
    {"a": 4,  "b": 7, "op": "multiply"},
    {"a": 20, "b": 4, "op": "divide"},
    {"a": 5,  "b": 5, "op": "add"},
]

for i, op_def in enumerate(operations, 1):
    state = {
        "a": op_def["a"],
        "b": op_def["b"],
        "operation": op_def["op"],
        "result": None,
        "history": history_so_far   # Pass accumulated history
    }
    out = graph.invoke(state)
    history_so_far = out["history"]  # Capture growing history
    print(f"  Call {i}: result={out['result']} | history length={len(history_so_far)}")

print()
print("── Full History ──────────────────────────────")
for i, entry in enumerate(history_so_far, 1):
    print(f"  {i}. {entry}")


## Custom Reducer — Fine-grained Control

You can write any reducer function:
```python
def reducer_fn(current_value, new_value):
    # Merge however you want
    return merged_result
```


In [ ]:
# ── Cell 8: Custom Reducer Demo — Keep Only Last 3 ──────────────────────────

def keep_last_3(current: list, update: list) -> list:
    """Keep only the most recent 3 history entries."""
    combined = current + update
    return combined[-3:]  # Sliding window of last 3

class State3(TypedDict):
    a: int
    b: int
    result: float
    history: Annotated[list, keep_last_3]  # Custom reducer

def add3(state: State3) -> dict:
    res = state["a"] + state["b"]
    return {"result": float(res), "history": [f"{state['a']}+{state['b']}={res}"]}

g3 = StateGraph(State3)
g3.add_node("add", add3)
g3.add_edge(START, "add")
g3.add_edge("add", END)
graph3 = g3.compile()

# Simulate 6 calls — only last 3 kept in history
h = []
for a, b in [(1,1),(2,2),(3,3),(4,4),(5,5),(6,6)]:
    out = graph3.invoke({"a":a,"b":b,"result":0.0,"history":h})
    h = out["history"]
    print(f"  After {a}+{b}: history = {h}")

print("\nNote: only the last 3 entries are kept (sliding window reducer)")


## How LangGraph Uses `Annotated`

```python
history: Annotated[list, operator.add]
#                  ↑        ↑
#                  type     reducer function
```

When LangGraph's state merger sees `Annotated[list, operator.add]`:
```python
# Instead of: new_state["history"] = node_return["history"]  (overwrite)
# It does:    new_state["history"] = operator.add(current["history"], node_return["history"])
```

This is exactly what `MessagesState` uses:
```python
# MessagesState in LangGraph source:
messages: Annotated[list[BaseMessage], add_messages]
#                                      ↑ special reducer that handles message deduplication
```

## Summary

| | Lesson 3 | Lesson 4 |
|--|----------|----------|
| State merge | Overwrite (last wins) | Reducer-controlled |
| History | Not tracked | Accumulated via `operator.add` |
| Parallel safety | No | Yes (reducer merges safely) |

## Limitations → What Lesson 5 Solves
- ❌ What if `operation` is `"subtract"` (unknown)?
- ❌ What if `b=0` for divide?
- ❌ We silently errored or used wrong defaults
- ❌ **Lesson 5**: Validation loops that **retry** until input is valid
